# Regresión lineal para predicción de progresión de diabetes
- age: Representa la edad del paciente, normalizada (no es la edad real, sino una representación estandarizada).
- sex: Sexo del paciente, representado como una variable numérica normalizada (valores entre -0.5 y 0.5).
- bmi: Índice de Masa Corporal (Body Mass Index, BMI), una medida del peso relativo al cuadrado de la altura.
- bp:
Promedio de presión arterial (Blood Pressure, BP), normalizada.
- s1: Nivel sérico de lípidos totales (colesterol total).
- s2: Nivel sérico de lipoproteínas de baja densidad (LDL, “colesterol malo”).
- s3: Nivel sérico de lipoproteínas de alta densidad (HDL, “colesterol bueno”).
- s4: Relación entre el colesterol total y HDL.
- s5: Nivel sérico de triglicéridos, representado como una medida numérica normalizada.
- s6: Nivel sérico de glucosa en ayuno, una medida clave en el monitoreo de la diabetes.

In [ ]:
# Importar las librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes

# Cargar el conjunto de datos
diabetes_data = load_diabetes()

# Convertir el conjunto de datos a un DataFrame para facilitar su manipulación
df = pd.DataFrame(data=diabetes_data.data, columns=diabetes_data.feature_names)
df['target'] = diabetes_data.target  # Agregar la variable objetivo al DataFrame

# Mostrar las primeras filas del DataFrame
df.head()

In [ ]:
# Exploración de los datos
print('Dimensiones del conjunto:', df.shape)
print('\nPrimeros registros:')
display(df.head())

print('\nInformación general:')
display(df.describe().round(2))

print('\nValores nulos por columna:')
display(df.isnull().sum())

# Separar las variables de entrada y la variable que se desea predecir
X = df.drop(columns='target')
y = df['target']

print('\nNúmero de características:', X.shape[1])
print('Número de registros:', len(X))


## Preparación y entrenamiento del modelo

Se separan los datos en entrenamiento y prueba. El 80 % se utiliza para entrenar y el 20 % para evaluar el modelo con datos que no ha visto. Las variables del conjunto Diabetes de scikit-learn ya vienen estandarizadas, por lo que se conservan sus valores originales.


In [ ]:
# Dividir los datos en entrenamiento y prueba
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Datos de entrenamiento:', X_train.shape[0])
print('Datos de prueba:', X_test.shape[0])

## Entrenamiento

Se utiliza una regresión lineal para relacionar las características médicas con la progresión de la diabetes.


In [ ]:
# Crear y entrenar el modelo
modelo = LinearRegression()
modelo.fit(X_train, y_train)

# Obtener las predicciones para los datos de prueba
y_pred = modelo.predict(X_test)

print('Modelo entrenado correctamente.')

## Evaluación del modelo

Se calculan tres métricas para conocer el desempeño de las predicciones:
- **MAE:** error absoluto medio; indica cuánto se alejan, en promedio, las predicciones de los valores reales.
- **RMSE:** raíz del error cuadrático medio; penaliza más los errores grandes.
- **R²:** proporción de la variación de la variable objetivo que explica el modelo.


In [ ]:
# Calcular las métricas de evaluación
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

resultados = pd.DataFrame({
    'Métrica': ['MAE', 'RMSE', 'R²'],
    'Resultado': [mae, rmse, r2]
})

display(resultados.round(4))

print(f'MAE: {mae:.2f}')
print(f'RMSE: {rmse:.2f}')
print(f'R²: {r2:.4f}')

## Gráfica de evaluación

La gráfica compara los valores reales con los valores estimados. Cuanto más cerca estén los puntos de la línea diagonal, mejores serán las predicciones.


In [ ]:
# Comparar valores reales y predichos
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.7)
limites = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(limites, limites, 'r--', label='Predicción ideal')
plt.xlabel('Progresión real')
plt.ylabel('Progresión predicha')
plt.title('Regresión lineal: valores reales vs. predichos')
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig('evaluacion_modelo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Revisar los coeficientes del modelo
coeficientes = pd.DataFrame({
    'Característica': X.columns,
    'Coeficiente': modelo.coef_
}).sort_values('Coeficiente', key=abs, ascending=False)

display(coeficientes.round(4))
print(f'Intercepto: {modelo.intercept_:.4f}')

## Resultado

El modelo genera una estimación de la progresión de la diabetes a partir de las diez características médicas. El valor de R² permite valorar qué parte de la variación de los datos de prueba logra explicar el modelo. Las métricas y la gráfica anteriores resumen su desempeño; los resultados corresponden a este conjunto de prueba y no deben interpretarse como un diagnóstico médico.
